# face_qef GPU Compare

JIT compile current `intersect_qef`, `face_qef_old`, `face_qef_ref`, current `face_qef`, and a CPU `face_qef` wrapper. `intersect_qef` only prepares the active voxel and brick inputs used by the face QEF variants.

In [1]:
import os
import sys
import time
import tempfile
import statistics
from pathlib import Path

CONDA_PREFIX = Path(os.environ.get('CONDA_PREFIX', '/home/quanta/.conda/envs/symm-enforce'))
os.environ['PATH'] = f"{CONDA_PREFIX / 'bin'}:{os.environ.get('PATH', '')}"
os.environ['CUDA_HOME'] = str(CONDA_PREFIX)
os.environ['CC'] = str(CONDA_PREFIX / 'bin/gcc')
os.environ['CXX'] = str(CONDA_PREFIX / 'bin/g++')
os.environ['MAX_JOBS'] = str(os.cpu_count() or 1)

import torch
import torch.utils.cpp_extension as cpp_extension
from torch.utils.cpp_extension import load

cpp_extension.CUDA_HOME = str(CONDA_PREFIX)
torch.set_grad_enabled(False)

ROOT = Path.cwd()
if not (ROOT / 'setup.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'setup.py').exists(), f'Cannot find repo root from {Path.cwd()}'

DEVICE = torch.device('cuda')
assert torch.cuda.is_available(), 'CUDA is required for this notebook'

GRID_SIZE = int(os.environ.get('FDG_GRID_SIZE', '512'))
CHUNK_TRIANGLES = int(os.environ.get('FDG_CHUNK_TRIANGLES', '20000'))
SUBDIVIDE_AREA_THRESHOLD = float(os.environ.get('FDG_SUBDIVIDE_AREA_THRESHOLD', str(1.0 / (GRID_SIZE * GRID_SIZE))))
SUBDIVIDE_MAX_ITERS = int(os.environ.get('FDG_SUBDIVIDE_MAX_ITERS', '12'))
WARM_REPEATS = int(os.environ.get('FDG_WARM_REPEATS', '10'))
RUN_CPU_SUBDIVIDED = os.environ.get('FDG_RUN_CPU_SUBDIVIDED', '1') != '0'

print('repo:', ROOT)
print('python:', sys.executable)
print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
print('device:', torch.cuda.get_device_name(DEVICE))
print('CONDA_PREFIX:', CONDA_PREFIX)
print('nvcc:', CONDA_PREFIX / 'bin/nvcc')
print('gcc:', os.environ['CC'])
print('g++:', os.environ['CXX'])
print('MAX_JOBS:', os.environ['MAX_JOBS'])
print('GRID_SIZE:', GRID_SIZE, 'CHUNK_TRIANGLES:', CHUNK_TRIANGLES)
print('SUBDIVIDE_AREA_THRESHOLD:', SUBDIVIDE_AREA_THRESHOLD, 'SUBDIVIDE_MAX_ITERS:', SUBDIVIDE_MAX_ITERS)
print('WARM_REPEATS:', WARM_REPEATS, 'RUN_CPU_SUBDIVIDED:', RUN_CPU_SUBDIVIDED)


repo: /mnt/nvmefs/Projects/o-voxel-gpu
python: /home/quanta/.conda/envs/symm-enforce/bin/python
torch: 2.6.0+cu124 cuda: 12.4
device: NVIDIA GeForce RTX 4090
CONDA_PREFIX: /home/quanta/.conda/envs/symm-enforce
nvcc: /home/quanta/.conda/envs/symm-enforce/bin/nvcc
gcc: /home/quanta/.conda/envs/symm-enforce/bin/gcc
g++: /home/quanta/.conda/envs/symm-enforce/bin/g++
MAX_JOBS: 32
GRID_SIZE: 512 CHUNK_TRIANGLES: 20000
SUBDIVIDE_AREA_THRESHOLD: 3.814697265625e-06 SUBDIVIDE_MAX_ITERS: 12
WARM_REPEATS: 10 RUN_CPU_SUBDIVIDED: True


## JIT Extension

The temporary C++ binding includes the CPU `face_qef` implementation from the current CPU source and binds the three GPU face QEF variants from the CUDA module.

In [2]:
build_dir = Path(tempfile.mkdtemp(prefix='fdg_face_qef_jit_'))
binding_path = build_dir / 'binding.cpp'
probe_path = build_dir / 'probe.cu'
torch_build_dir = build_dir / 'torch_ext'
torch_build_dir.mkdir(parents=True, exist_ok=True)

binding_cpp = r"""
#include <torch/extension.h>
#include <Eigen/Dense>
#include <unordered_map>
#include <vector>
#include "__ROOT__/src/convert/flexible_dual_grid.cpp"
#include "__ROOT__/src/convert/mesh_to_flexible_dual_grid_gpu/api.h"

namespace py = pybind11;

torch::Tensor face_qef_gpu_hit_probe(
    const torch::Tensor& triangles,
    const torch::Tensor& voxel_size,
    const torch::Tensor& grid_range,
    const torch::Tensor& voxel);

namespace {

torch::Tensor face_qef_cpu_only(
    const torch::Tensor& triangles,
    const torch::Tensor& voxel_size,
    const torch::Tensor& grid_range,
    const torch::Tensor& voxels) {
    const int64_t T = triangles.size(0);
    const int64_t N = voxels.size(0);
    const float* tri_ptr = triangles.data_ptr<float>();
    const float* vs = voxel_size.data_ptr<float>();
    const int32_t* gr = grid_range.data_ptr<int32_t>();
    const int32_t* vox_ptr = voxels.data_ptr<int32_t>();

    std::vector<Eigen::Vector3f> tri_vec;
    tri_vec.reserve(static_cast<size_t>(T) * 3);
    for (int64_t t = 0; t < T; ++t) {
        tri_vec.emplace_back(tri_ptr[9 * t + 0], tri_ptr[9 * t + 1], tri_ptr[9 * t + 2]);
        tri_vec.emplace_back(tri_ptr[9 * t + 3], tri_ptr[9 * t + 4], tri_ptr[9 * t + 5]);
        tri_vec.emplace_back(tri_ptr[9 * t + 6], tri_ptr[9 * t + 7], tri_ptr[9 * t + 8]);
    }

    std::unordered_map<VoxelCoord, size_t> hash_table;
    hash_table.reserve(static_cast<size_t>(N) * 2);
    for (int64_t i = 0; i < N; ++i) {
        hash_table[VoxelCoord{vox_ptr[3 * i + 0], vox_ptr[3 * i + 1], vox_ptr[3 * i + 2]}] = static_cast<size_t>(i);
    }

    std::vector<Eigen::Matrix4f> qefs(static_cast<size_t>(N), Eigen::Matrix4f::Zero());
    ::face_qef(
        Eigen::Vector3f(vs[0], vs[1], vs[2]),
        Eigen::Vector3i(gr[0], gr[1], gr[2]),
        Eigen::Vector3i(gr[3], gr[4], gr[5]),
        tri_vec,
        hash_table,
        qefs);

    auto out = torch::empty({N, 10}, torch::TensorOptions().dtype(torch::kFloat32).device(torch::kCPU));
    float* out_ptr = out.data_ptr<float>();
    for (int64_t i = 0; i < N; ++i) {
        const Eigen::Matrix4f& Q = qefs[static_cast<size_t>(i)];
        out_ptr[10 * i + 0] = Q(0, 0);
        out_ptr[10 * i + 1] = Q(0, 1);
        out_ptr[10 * i + 2] = Q(0, 2);
        out_ptr[10 * i + 3] = Q(0, 3);
        out_ptr[10 * i + 4] = Q(1, 1);
        out_ptr[10 * i + 5] = Q(1, 2);
        out_ptr[10 * i + 6] = Q(1, 3);
        out_ptr[10 * i + 7] = Q(2, 2);
        out_ptr[10 * i + 8] = Q(2, 3);
        out_ptr[10 * i + 9] = Q(3, 3);
    }
    return out;
}


torch::Tensor face_qef_cpu_hit_probe(
    const torch::Tensor& triangles,
    const torch::Tensor& voxel_size,
    const torch::Tensor& grid_range,
    const torch::Tensor& voxel) {
    const int64_t T = triangles.size(0);
    const float* tri = triangles.data_ptr<float>();
    const float* vs = voxel_size.data_ptr<float>();
    const int32_t* gr = grid_range.data_ptr<int32_t>();
    const int32_t* vp = voxel.data_ptr<int32_t>();
    const int x = vp[0];
    const int y = vp[1];
    const int z = vp[2];

    const Eigen::Vector3f e_voxel_size(vs[0], vs[1], vs[2]);
    const Eigen::Vector3i grid_min(gr[0], gr[1], gr[2]);
    const Eigen::Vector3i grid_max(gr[3], gr[4], gr[5]);
    const Eigen::Vector3f p = e_voxel_size.cwiseProduct(Eigen::Vector3f(x, y, z));

    auto out = torch::zeros({T}, torch::TensorOptions().dtype(torch::kUInt8).device(torch::kCPU));
    uint8_t* out_ptr = out.data_ptr<uint8_t>();

    for (int64_t t = 0; t < T; ++t) {
        const Eigen::Vector3f v0(tri[9 * t + 0], tri[9 * t + 1], tri[9 * t + 2]);
        const Eigen::Vector3f v1(tri[9 * t + 3], tri[9 * t + 4], tri[9 * t + 5]);
        const Eigen::Vector3f v2(tri[9 * t + 6], tri[9 * t + 7], tri[9 * t + 8]);
        const Eigen::Vector3f e0 = v1 - v0;
        const Eigen::Vector3f e1 = v2 - v1;
        const Eigen::Vector3f e2 = v0 - v2;
        const Eigen::Vector3f n = e0.cross(e1).normalized();

        const Eigen::Vector3f bb_min_f = v0.cwiseMin(v1).cwiseMin(v2).cwiseQuotient(e_voxel_size);
        const Eigen::Vector3f bb_max_f = v0.cwiseMax(v1).cwiseMax(v2).cwiseQuotient(e_voxel_size);
        const Eigen::Vector3i bb_min(std::max(static_cast<int>(bb_min_f.x()), grid_min.x()),
                                     std::max(static_cast<int>(bb_min_f.y()), grid_min.y()),
                                     std::max(static_cast<int>(bb_min_f.z()), grid_min.z()));
        const Eigen::Vector3i bb_max(std::min(static_cast<int>(bb_max_f.x() + 1), grid_max.x()),
                                     std::min(static_cast<int>(bb_max_f.y() + 1), grid_max.y()),
                                     std::min(static_cast<int>(bb_max_f.z() + 1), grid_max.z()));
        if (x < bb_min.x() || x >= bb_max.x() || y < bb_min.y() || y >= bb_max.y() || z < bb_min.z() || z >= bb_max.z())
            continue;

        const Eigen::Vector3f c(
            n.x() > 0.0f ? e_voxel_size.x() : 0.0f,
            n.y() > 0.0f ? e_voxel_size.y() : 0.0f,
            n.z() > 0.0f ? e_voxel_size.z() : 0.0f);
        const float d1 = n.dot(c - v0);
        const float d2 = n.dot(e_voxel_size - c - v0);
        const float n_dot_p = n.dot(p);
        if (((n_dot_p + d1) * (n_dot_p + d2)) > 0.0f)
            continue;

        const int mul_xy = (n.z() < 0.0f) ? -1 : 1;
        const Eigen::Vector2f n_xy_e0(-mul_xy * e0.y(), mul_xy * e0.x());
        const Eigen::Vector2f n_xy_e1(-mul_xy * e1.y(), mul_xy * e1.x());
        const Eigen::Vector2f n_xy_e2(-mul_xy * e2.y(), mul_xy * e2.x());
        const float d_xy_e0 = -n_xy_e0.dot(v0.head<2>()) + n_xy_e0.cwiseMax(0.0f).dot(e_voxel_size.head<2>());
        const float d_xy_e1 = -n_xy_e1.dot(v1.head<2>()) + n_xy_e1.cwiseMax(0.0f).dot(e_voxel_size.head<2>());
        const float d_xy_e2 = -n_xy_e2.dot(v2.head<2>()) + n_xy_e2.cwiseMax(0.0f).dot(e_voxel_size.head<2>());
        if (n_xy_e0.dot(p.head<2>()) + d_xy_e0 < 0.0f) continue;
        if (n_xy_e1.dot(p.head<2>()) + d_xy_e1 < 0.0f) continue;
        if (n_xy_e2.dot(p.head<2>()) + d_xy_e2 < 0.0f) continue;

        const int mul_yz = (n.x() < 0.0f) ? -1 : 1;
        const Eigen::Vector2f n_yz_e0(-mul_yz * e0.z(), mul_yz * e0.y());
        const Eigen::Vector2f n_yz_e1(-mul_yz * e1.z(), mul_yz * e1.y());
        const Eigen::Vector2f n_yz_e2(-mul_yz * e2.z(), mul_yz * e2.y());
        const Eigen::Vector2f p_yz(p.y(), p.z());
        const float d_yz_e0 = -n_yz_e0.dot(Eigen::Vector2f(v0.y(), v0.z())) + n_yz_e0.cwiseMax(0.0f).dot(Eigen::Vector2f(e_voxel_size.y(), e_voxel_size.z()));
        const float d_yz_e1 = -n_yz_e1.dot(Eigen::Vector2f(v1.y(), v1.z())) + n_yz_e1.cwiseMax(0.0f).dot(Eigen::Vector2f(e_voxel_size.y(), e_voxel_size.z()));
        const float d_yz_e2 = -n_yz_e2.dot(Eigen::Vector2f(v2.y(), v2.z())) + n_yz_e2.cwiseMax(0.0f).dot(Eigen::Vector2f(e_voxel_size.y(), e_voxel_size.z()));
        if (n_yz_e0.dot(p_yz) + d_yz_e0 < 0.0f) continue;
        if (n_yz_e1.dot(p_yz) + d_yz_e1 < 0.0f) continue;
        if (n_yz_e2.dot(p_yz) + d_yz_e2 < 0.0f) continue;

        const int mul_zx = (n.y() < 0.0f) ? -1 : 1;
        const Eigen::Vector2f n_zx_e0(-mul_zx * e0.x(), mul_zx * e0.z());
        const Eigen::Vector2f n_zx_e1(-mul_zx * e1.x(), mul_zx * e1.z());
        const Eigen::Vector2f n_zx_e2(-mul_zx * e2.x(), mul_zx * e2.z());
        const Eigen::Vector2f p_zx(p.z(), p.x());
        const float d_zx_e0 = -n_zx_e0.dot(Eigen::Vector2f(v0.z(), v0.x())) + n_zx_e0.cwiseMax(0.0f).dot(Eigen::Vector2f(e_voxel_size.z(), e_voxel_size.x()));
        const float d_zx_e1 = -n_zx_e1.dot(Eigen::Vector2f(v1.z(), v1.x())) + n_zx_e1.cwiseMax(0.0f).dot(Eigen::Vector2f(e_voxel_size.z(), e_voxel_size.x()));
        const float d_zx_e2 = -n_zx_e2.dot(Eigen::Vector2f(v2.z(), v2.x())) + n_zx_e2.cwiseMax(0.0f).dot(Eigen::Vector2f(e_voxel_size.z(), e_voxel_size.x()));
        if (n_zx_e0.dot(p_zx) + d_zx_e0 < 0.0f) continue;
        if (n_zx_e1.dot(p_zx) + d_zx_e1 < 0.0f) continue;
        if (n_zx_e2.dot(p_zx) + d_zx_e2 < 0.0f) continue;

        out_ptr[t] = 1;
    }
    return out;
}

} // namespace

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("intersect_qef", &o_voxel::fdg::intersect_qef);
    m.def("face_qef_old", &o_voxel::fdg::face_qef_old);
    m.def("face_qef_ref", &o_voxel::fdg::face_qef_ref);
    m.def("face_qef", &o_voxel::fdg::face_qef);
    m.def("face_qef_cpu", &face_qef_cpu_only);
    m.def("face_qef_cpu_hit_probe", &face_qef_cpu_hit_probe);
    m.def("face_qef_gpu_hit_probe", &face_qef_gpu_hit_probe);
}
""".replace('__ROOT__', ROOT.as_posix())


probe_cu = r"""
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAGuard.h>
#include <c10/cuda/CUDAException.h>
#include <cuda_runtime.h>
#include "mesh_to_flexible_dual_grid_gpu/types.cuh"

using o_voxel::fdg::Int3;

namespace {

constexpr int kThreads = 256;

__device__ __forceinline__ float pmin(float a, float b) { return a < b ? a : b; }
__device__ __forceinline__ float pmax(float a, float b) { return a > b ? a : b; }

__device__ __forceinline__ float3 aligned_cross_normalize3(
    float e0x,
    float e0y,
    float e0z,
    float e1x,
    float e1y,
    float e1z) {
    const float nx = __fsub_rn(__fmul_rn(e0y, e1z), __fmul_rn(e0z, e1y));
    const float ny = __fsub_rn(__fmul_rn(e0z, e1x), __fmul_rn(e0x, e1z));
    const float nz = __fsub_rn(__fmul_rn(e0x, e1y), __fmul_rn(e0y, e1x));
    const float n2 = __fadd_rn(
        __fadd_rn(__fmul_rn(nx, nx), __fmul_rn(ny, ny)),
        __fmul_rn(nz, nz));
    if (n2 > 0.0f) {
        const float len = sqrtf(n2);
        return make_float3(nx / len, ny / len, nz / len);
    }
    return make_float3(nx, ny, nz);
}

__global__ void face_qef_hit_kernel(
    int64_t num_triangles,
    const float* __restrict__ triangles,
    float3 voxel_size,
    Int3 grid_min,
    Int3 grid_max,
    int x,
    int y,
    int z,
    uint8_t* __restrict__ hit) {
    const int64_t tid = static_cast<int64_t>(blockIdx.x) * blockDim.x + threadIdx.x;
    if (tid >= num_triangles)
        return;

    const float* tri = triangles + tid * 9;
    const float3 v0 = make_float3(tri[0], tri[1], tri[2]);
    const float3 v1 = make_float3(tri[3], tri[4], tri[5]);
    const float3 v2 = make_float3(tri[6], tri[7], tri[8]);
    const float vs[3] = {voxel_size.x, voxel_size.y, voxel_size.z};

    const float e0x = v1.x - v0.x;
    const float e0y = v1.y - v0.y;
    const float e0z = v1.z - v0.z;
    const float e1x = v2.x - v1.x;
    const float e1y = v2.y - v1.y;
    const float e1z = v2.z - v1.z;
    const float e2x = v0.x - v2.x;
    const float e2y = v0.y - v2.y;
    const float e2z = v0.z - v2.z;

    const float3 n = aligned_cross_normalize3(e0x, e0y, e0z, e1x, e1y, e1z);
    const float nx = n.x;
    const float ny = n.y;
    const float nz = n.z;

    const float bb_min_f_x = pmin(pmin(v0.x, v1.x), v2.x) / vs[0];
    const float bb_min_f_y = pmin(pmin(v0.y, v1.y), v2.y) / vs[1];
    const float bb_min_f_z = pmin(pmin(v0.z, v1.z), v2.z) / vs[2];
    const float bb_max_f_x = pmax(pmax(v0.x, v1.x), v2.x) / vs[0];
    const float bb_max_f_y = pmax(pmax(v0.y, v1.y), v2.y) / vs[1];
    const float bb_max_f_z = pmax(pmax(v0.z, v1.z), v2.z) / vs[2];
    const int bb_min_x = max(static_cast<int>(bb_min_f_x), grid_min.x);
    const int bb_min_y = max(static_cast<int>(bb_min_f_y), grid_min.y);
    const int bb_min_z = max(static_cast<int>(bb_min_f_z), grid_min.z);
    const int bb_max_x = min(static_cast<int>(bb_max_f_x + 1.0f), grid_max.x);
    const int bb_max_y = min(static_cast<int>(bb_max_f_y + 1.0f), grid_max.y);
    const int bb_max_z = min(static_cast<int>(bb_max_f_z + 1.0f), grid_max.z);
    if (x < bb_min_x || x >= bb_max_x || y < bb_min_y || y >= bb_max_y || z < bb_min_z || z >= bb_max_z)
        return;

    const float px = x * vs[0];
    const float py = y * vs[1];
    const float pz = z * vs[2];
    const float c_x = nx > 0.0f ? vs[0] : 0.0f;
    const float c_y = ny > 0.0f ? vs[1] : 0.0f;
    const float c_z = nz > 0.0f ? vs[2] : 0.0f;
    const float d1 = nx * (c_x - v0.x) + ny * (c_y - v0.y) + nz * (c_z - v0.z);
    const float d2 = nx * (vs[0] - c_x - v0.x) + ny * (vs[1] - c_y - v0.y) + nz * (vs[2] - c_z - v0.z);
    const float n_dot_p = nx * px + ny * py + nz * pz;
    if ((n_dot_p + d1) * (n_dot_p + d2) > 0.0f)
        return;

    const int mul_xy = nz < 0.0f ? -1 : 1;
    const float n_xy_e0_x = -mul_xy * e0y;
    const float n_xy_e0_y = mul_xy * e0x;
    const float n_xy_e1_x = -mul_xy * e1y;
    const float n_xy_e1_y = mul_xy * e1x;
    const float n_xy_e2_x = -mul_xy * e2y;
    const float n_xy_e2_y = mul_xy * e2x;
    const float d_xy_e0 = -(n_xy_e0_x * v0.x + n_xy_e0_y * v0.y) + fmaxf(n_xy_e0_x, 0.0f) * vs[0] + fmaxf(n_xy_e0_y, 0.0f) * vs[1];
    const float d_xy_e1 = -(n_xy_e1_x * v1.x + n_xy_e1_y * v1.y) + fmaxf(n_xy_e1_x, 0.0f) * vs[0] + fmaxf(n_xy_e1_y, 0.0f) * vs[1];
    const float d_xy_e2 = -(n_xy_e2_x * v2.x + n_xy_e2_y * v2.y) + fmaxf(n_xy_e2_x, 0.0f) * vs[0] + fmaxf(n_xy_e2_y, 0.0f) * vs[1];
    if (n_xy_e0_x * px + n_xy_e0_y * py + d_xy_e0 < 0.0f) return;
    if (n_xy_e1_x * px + n_xy_e1_y * py + d_xy_e1 < 0.0f) return;
    if (n_xy_e2_x * px + n_xy_e2_y * py + d_xy_e2 < 0.0f) return;

    const int mul_yz = nx < 0.0f ? -1 : 1;
    const float n_yz_e0_x = -mul_yz * e0z;
    const float n_yz_e0_y = mul_yz * e0y;
    const float n_yz_e1_x = -mul_yz * e1z;
    const float n_yz_e1_y = mul_yz * e1y;
    const float n_yz_e2_x = -mul_yz * e2z;
    const float n_yz_e2_y = mul_yz * e2y;
    const float d_yz_e0 = -(n_yz_e0_x * v0.y + n_yz_e0_y * v0.z) + fmaxf(n_yz_e0_x, 0.0f) * vs[1] + fmaxf(n_yz_e0_y, 0.0f) * vs[2];
    const float d_yz_e1 = -(n_yz_e1_x * v1.y + n_yz_e1_y * v1.z) + fmaxf(n_yz_e1_x, 0.0f) * vs[1] + fmaxf(n_yz_e1_y, 0.0f) * vs[2];
    const float d_yz_e2 = -(n_yz_e2_x * v2.y + n_yz_e2_y * v2.z) + fmaxf(n_yz_e2_x, 0.0f) * vs[1] + fmaxf(n_yz_e2_y, 0.0f) * vs[2];
    if (n_yz_e0_x * py + n_yz_e0_y * pz + d_yz_e0 < 0.0f) return;
    if (n_yz_e1_x * py + n_yz_e1_y * pz + d_yz_e1 < 0.0f) return;
    if (n_yz_e2_x * py + n_yz_e2_y * pz + d_yz_e2 < 0.0f) return;

    const int mul_zx = ny < 0.0f ? -1 : 1;
    const float n_zx_e0_x = -mul_zx * e0x;
    const float n_zx_e0_y = mul_zx * e0z;
    const float n_zx_e1_x = -mul_zx * e1x;
    const float n_zx_e1_y = mul_zx * e1z;
    const float n_zx_e2_x = -mul_zx * e2x;
    const float n_zx_e2_y = mul_zx * e2z;
    const float d_zx_e0 = -(n_zx_e0_x * v0.z + n_zx_e0_y * v0.x) + fmaxf(n_zx_e0_x, 0.0f) * vs[2] + fmaxf(n_zx_e0_y, 0.0f) * vs[0];
    const float d_zx_e1 = -(n_zx_e1_x * v1.z + n_zx_e1_y * v1.x) + fmaxf(n_zx_e1_x, 0.0f) * vs[2] + fmaxf(n_zx_e1_y, 0.0f) * vs[0];
    const float d_zx_e2 = -(n_zx_e2_x * v2.z + n_zx_e2_y * v2.x) + fmaxf(n_zx_e2_x, 0.0f) * vs[2] + fmaxf(n_zx_e2_y, 0.0f) * vs[0];
    if (n_zx_e0_x * pz + n_zx_e0_y * px + d_zx_e0 < 0.0f) return;
    if (n_zx_e1_x * pz + n_zx_e1_y * px + d_zx_e1 < 0.0f) return;
    if (n_zx_e2_x * pz + n_zx_e2_y * px + d_zx_e2 < 0.0f) return;

    hit[tid] = 1;
}

} // namespace

torch::Tensor face_qef_gpu_hit_probe(
    const torch::Tensor& triangles,
    const torch::Tensor& voxel_size,
    const torch::Tensor& grid_range,
    const torch::Tensor& voxel) {
    const c10::cuda::CUDAGuard guard(triangles.device());
    const cudaStream_t stream = at::cuda::getCurrentCUDAStream(triangles.get_device()).stream();
    const int64_t T = triangles.size(0);
    auto hit = torch::zeros({T}, torch::TensorOptions().dtype(torch::kUInt8).device(triangles.device()));
    if (T == 0)
        return hit;

    const float* vs = voxel_size.data_ptr<float>();
    const int32_t* gr = grid_range.data_ptr<int32_t>();
    const int32_t* vp = voxel.data_ptr<int32_t>();
    const float3 voxel_size_v = make_float3(vs[0], vs[1], vs[2]);
    const Int3 grid_min{gr[0], gr[1], gr[2]};
    const Int3 grid_max{gr[3], gr[4], gr[5]};
    const int blocks = static_cast<int>((T + kThreads - 1) / kThreads);
    face_qef_hit_kernel<<<blocks, kThreads, 0, stream>>>(
        T,
        triangles.data_ptr<float>(),
        voxel_size_v,
        grid_min,
        grid_max,
        vp[0],
        vp[1],
        vp[2],
        hit.data_ptr<uint8_t>());
    C10_CUDA_KERNEL_LAUNCH_CHECK();
    return hit;
}
"""

binding_path.write_text(binding_cpp)
probe_path.write_text(probe_cu)

sources = [
    str(binding_path),
    str(probe_path),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/intersect_qef.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/voxelize_mesh_octree.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/face_qef_old.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/face_qef_ref.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/face_qef.cu'),
]

ext = load(
    name=f'fdg_face_qef_jit_{os.getpid()}',
    sources=sources,
    build_directory=str(torch_build_dir),
    extra_include_paths=[
        str(ROOT / 'src/convert'),
        str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu'),
        str(ROOT / 'third_party/eigen'),
    ],
    extra_cflags=['-O3', '-std=c++17'],
    extra_cuda_cflags=['-O3', '-std=c++17'],
    with_cuda=True,
    verbose=True,
)
print('JIT extension loaded:', ext)


Detected CUDA files, patching ldflags
Emitting ninja build file /tmp/fdg_face_qef_jit_uwlzru4y/torch_ext/build.ninja...
/home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fdg_face_qef_jit_1153316...
Using envvar MAX_JOBS (32) as the number of workers...


[1/8] /home/quanta/.conda/envs/symm-enforce/bin/g++ -MMD -MF binding.o.d -DTORCH_EXTENSION_NAME=fdg_face_qef_jit_1153316 -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -I/mnt/nvmefs/Projects/o-voxel-gpu/src/convert -I/mnt/nvmefs/Projects/o-voxel-gpu/src/convert/mesh_to_flexible_dual_grid_gpu -I/mnt/nvmefs/Projects/o-voxel-gpu/third_party/eigen -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/torch/csrc/api/include -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/TH -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/THC -isystem /home/quanta/.conda/envs/symm-enforce/include -isystem /home/quanta/.conda/envs/symm-enforce/include/python3.10 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -O3 -std=c

Loading extension module fdg_face_qef_jit_1153316...


## Load Mesh Inputs

The original mesh comes from `notebooks/test.glb`. The subdivided input uses the same GPU subdivision helper as the intersect notebook.

In [3]:
import numpy as np
import trimesh


def subdivide_mesh_gpu(verts: torch.Tensor, faces: torch.Tensor, area_threshold: float, max_iters: int):
    from pytorch3d.structures import Meshes

    device = verts.device
    cur = Meshes(verts=[verts], faces=[faces])

    for _ in range(max_iters):
        cv = cur.verts_packed()
        cf = cur.faces_packed()
        edges = cur.edges_packed()
        f2e = cur.faces_packed_to_edges_packed()

        areas = cur.faces_areas_packed()
        large = areas > area_threshold
        if not large.any():
            break

        edge_len = torch.norm(cv[edges[:, 0]] - cv[edges[:, 1]], dim=1)
        _, local_max = torch.max(edge_len[f2e], dim=1)
        max_edge_ids = f2e[torch.arange(cf.shape[0], device=device), local_max]

        target = torch.zeros(edges.shape[0], dtype=torch.bool, device=device)
        target[max_edge_ids[large]] = True
        if not target.any():
            break

        mid = (cv[edges[target, 0]] + cv[edges[target, 1]]) * 0.5
        new_verts = torch.cat([cv, mid], dim=0)
        n_old = cv.shape[0]

        e2new = torch.zeros(edges.shape[0], dtype=torch.long, device=device)
        e2new[target] = torch.arange(n_old, n_old + target.sum().item(), device=device)

        split_counts = target[f2e].sum(dim=1)

        v0, v1, v2 = cf[:, 0], cf[:, 1], cf[:, 2]
        p01_a = torch.minimum(v0, v1)
        p01_b = torch.maximum(v0, v1)
        p12_a = torch.minimum(v1, v2)
        p12_b = torch.maximum(v1, v2)
        p20_a = torch.minimum(v2, v0)
        p20_b = torch.maximum(v2, v0)

        e_act = edges[f2e]
        match01 = (e_act[:, :, 0] == p01_a.unsqueeze(1)) & (e_act[:, :, 1] == p01_b.unsqueeze(1))
        match12 = (e_act[:, :, 0] == p12_a.unsqueeze(1)) & (e_act[:, :, 1] == p12_b.unsqueeze(1))
        match20 = (e_act[:, :, 0] == p20_a.unsqueeze(1)) & (e_act[:, :, 1] == p20_b.unsqueeze(1))

        col01 = match01.long().argmax(dim=1)
        col12 = match12.long().argmax(dim=1)
        col20 = match20.long().argmax(dim=1)

        keep = cf[split_counts == 0]
        out = [keep]

        m1 = split_counts == 1
        if m1.any():
            f1 = cf[m1]
            fe1 = f2e[m1]
            M = f1.shape[0]
            v0_1, v1_1, v2_1 = f1[:, 0], f1[:, 1], f1[:, 2]

            split_col = target[fe1].long().argmax(dim=1)
            c01_m1 = col01[m1]
            c12_m1 = col12[m1]
            c20_m1 = col20[m1]
            is_split_01 = split_col == c01_m1
            is_split_12 = split_col == c12_m1
            is_split_20 = split_col == c20_m1

            split_eid = fe1[torch.arange(M, device=device), split_col]
            vn = e2new[split_eid]

            s1 = torch.zeros(M, 3, dtype=torch.long, device=device)
            s2 = torch.zeros(M, 3, dtype=torch.long, device=device)

            s1[is_split_01] = torch.stack([v0_1[is_split_01], vn[is_split_01], v2_1[is_split_01]], dim=1)
            s2[is_split_01] = torch.stack([vn[is_split_01], v1_1[is_split_01], v2_1[is_split_01]], dim=1)
            s1[is_split_12] = torch.stack([v0_1[is_split_12], v1_1[is_split_12], vn[is_split_12]], dim=1)
            s2[is_split_12] = torch.stack([v0_1[is_split_12], vn[is_split_12], v2_1[is_split_12]], dim=1)
            s1[is_split_20] = torch.stack([v0_1[is_split_20], v1_1[is_split_20], vn[is_split_20]], dim=1)
            s2[is_split_20] = torch.stack([vn[is_split_20], v1_1[is_split_20], v2_1[is_split_20]], dim=1)

            out.extend([s1, s2])

        mm = split_counts >= 2
        if mm.any():
            fm = cf[mm]
            fem = f2e[mm]
            M2 = fm.shape[0]
            v0_m, v1_m, v2_m = fm[:, 0], fm[:, 1], fm[:, 2]

            c01_m = col01[mm]
            c12_m = col12[mm]
            c20_m = col20[mm]

            eid01 = fem[torch.arange(M2, device=device), c01_m]
            e01_is_split = target[eid01]
            e01_v = torch.where(e01_is_split, e2new[eid01], v0_m)

            eid12 = fem[torch.arange(M2, device=device), c12_m]
            e12_is_split = target[eid12]
            e12_v = torch.where(e12_is_split, e2new[eid12], v1_m)

            eid20 = fem[torch.arange(M2, device=device), c20_m]
            e20_is_split = target[eid20]
            e20_v = torch.where(e20_is_split, e2new[eid20], v2_m)

            mf1 = torch.stack([v0_m, e01_v, e20_v], dim=1)
            mf2 = torch.stack([e01_v, v1_m, e12_v], dim=1)
            mf3 = torch.stack([e20_v, e12_v, v2_m], dim=1)
            mf4 = torch.stack([e01_v, e12_v, e20_v], dim=1)
            out.extend([mf1, mf2, mf3, mf4])

        cur = Meshes(verts=[new_verts], faces=[torch.cat(out, dim=0)])

    return cur.verts_packed(), cur.faces_packed()


def triangle_mb(triangles: torch.Tensor) -> float:
    return triangles.numel() * triangles.element_size() / 1024**2


glb_path = ROOT / 'notebooks/test.glb'
loaded = trimesh.load(glb_path, force='scene')
if isinstance(loaded, trimesh.Scene):
    meshes = [g for g in loaded.geometry.values() if isinstance(g, trimesh.Trimesh) and len(g.faces) > 0]
    mesh = trimesh.util.concatenate(meshes)
else:
    mesh = loaded
assert isinstance(mesh, trimesh.Trimesh) and len(mesh.faces) > 0

vertices_np = np.asarray(mesh.vertices, dtype=np.float32)
faces_np = np.asarray(mesh.faces, dtype=np.int64)
vmin = vertices_np.min(axis=0)
vmax = vertices_np.max(axis=0)
center = (vmin + vmax) * 0.5
scale = np.float32(0.999 / max((vmax - vmin).max(), 1e-12))
vertices_np = (vertices_np - center) * scale + np.float32(0.5)

vertices_cpu = torch.from_numpy(vertices_np).contiguous()
faces_cpu = torch.from_numpy(faces_np).contiguous()
triangles_cpu = vertices_cpu[faces_cpu].reshape(-1, 9).contiguous()
triangles_cuda = triangles_cpu.to(DEVICE, non_blocking=True).contiguous()

voxel_size_cpu = torch.tensor([1.0 / GRID_SIZE, 1.0 / GRID_SIZE, 1.0 / GRID_SIZE], dtype=torch.float32)
grid_range_cpu = torch.tensor([0, 0, 0, GRID_SIZE, GRID_SIZE, GRID_SIZE], dtype=torch.int32)

print('building subdivided input...')
torch.cuda.empty_cache()
torch.cuda.synchronize()
start = time.perf_counter()
sub_vertices_cuda, sub_faces_cuda = subdivide_mesh_gpu(
    vertices_cpu.to(DEVICE, non_blocking=True).contiguous(),
    faces_cpu.to(DEVICE, non_blocking=True).contiguous(),
    area_threshold=SUBDIVIDE_AREA_THRESHOLD,
    max_iters=SUBDIVIDE_MAX_ITERS,
)
sub_triangles_cuda = sub_vertices_cuda[sub_faces_cuda].reshape(-1, 9).contiguous()
torch.cuda.synchronize()
subdivide_ms = (time.perf_counter() - start) * 1000.0
sub_triangles_cpu = sub_triangles_cuda.detach().cpu().contiguous()
sub_vertices_shape = tuple(sub_vertices_cuda.shape)
sub_faces_shape = tuple(sub_faces_cuda.shape)
del sub_vertices_cuda, sub_faces_cuda
torch.cuda.empty_cache()

INPUT_CASES = {
    'original': {
        'triangles_cpu': triangles_cpu,
        'triangles_cuda': triangles_cuda,
        'vertices_shape': tuple(vertices_cpu.shape),
        'faces_shape': tuple(faces_cpu.shape),
        'triangles': int(triangles_cpu.shape[0]),
        'triangle_mb': triangle_mb(triangles_cuda),
        'preprocess_ms': 0.0,
    },
    'subdivided': {
        'triangles_cpu': sub_triangles_cpu,
        'triangles_cuda': sub_triangles_cuda,
        'vertices_shape': sub_vertices_shape,
        'faces_shape': sub_faces_shape,
        'triangles': int(sub_triangles_cpu.shape[0]),
        'triangle_mb': triangle_mb(sub_triangles_cuda),
        'preprocess_ms': subdivide_ms,
    },
}

for name, case in INPUT_CASES.items():
    print(
        name,
        'vertices:', case['vertices_shape'],
        'faces:', case['faces_shape'],
        'triangles:', case['triangles'],
        'triangle_mb:', case['triangle_mb'],
        'preprocess_ms:', case['preprocess_ms'],
    )
print('face multiplier:', float(INPUT_CASES['subdivided']['triangles'] / max(INPUT_CASES['original']['triangles'], 1)))
print('voxel_size:', voxel_size_cpu.tolist(), 'grid_range:', grid_range_cpu.tolist())


building subdivided input...
original vertices: (268018, 3) faces: (280333, 3) triangles: 280333 triangle_mb: 9.624469757080078 preprocess_ms: 0.0
subdivided vertices: (3608864, 3) faces: (10304077, 3) triangles: 10304077 triangle_mb: 353.7624092102051 preprocess_ms: 777.3909960014862
face multiplier: 36.75656094715927
voxel_size: [0.001953125, 0.001953125, 0.001953125] grid_range: [0, 0, 0, 512, 512, 512]


## Helpers

GPU functions take CUDA triangles. `voxel_size` and `grid_range` stay as CPU tensors because the C++ functions read their scalar values on host.

In [4]:
from typing import NamedTuple


class IntersectOutput(NamedTuple):
    voxels: torch.Tensor
    mean_sum: torch.Tensor
    cnt: torch.Tensor
    intersected: torch.Tensor
    qefs: torch.Tensor
    brick_hash_keys: torch.Tensor
    brick_hash_vals: torch.Tensor
    brick_bits: torch.Tensor
    brick_base: torch.Tensor


def as_intersect_output(output):
    assert len(output) == 9, f'expected 9 tensors from intersect_qef, got {len(output)}'
    return IntersectOutput(*output)


def voxel_key(voxels: torch.Tensor) -> torch.Tensor:
    voxels = voxels.to(torch.int64)
    rel = voxels - grid_range_cpu[:3].to(torch.int64)
    size = (grid_range_cpu[3:] - grid_range_cpu[:3]).to(torch.int64)
    return rel[:, 0] + size[0] * (rel[:, 1] + size[1] * rel[:, 2])


def canonical_face_qef(voxels: torch.Tensor, qefs: torch.Tensor):
    voxels_cpu = voxels.detach().cpu().contiguous()
    qefs_cpu = qefs.detach().cpu().float().contiguous()
    keys = voxel_key(voxels_cpu)
    order = torch.argsort(keys)
    keys = keys[order]
    unique = torch.unique_consecutive(keys).numel() == keys.numel()
    return {
        'keys': keys,
        'voxels': voxels_cpu[order],
        'qefs': qefs_cpu[order],
        'unique': bool(unique),
    }


def abs_distribution(prefix, values):
    flat = values.detach().float().abs().reshape(-1)
    if flat.numel() == 0:
        return {
            f'{prefix}_mean_abs': 0.0,
            f'{prefix}_p50_abs': 0.0,
            f'{prefix}_p90_abs': 0.0,
            f'{prefix}_p99_abs': 0.0,
            f'{prefix}_p999_abs': 0.0,
            f'{prefix}_max_abs': 0.0,
            f'{prefix}_gt_1e-5_ratio': 0.0,
            f'{prefix}_gt_1e-4_ratio': 0.0,
        }
    n = flat.numel()
    def pct(q):
        k = min(max(int(q * (n - 1)) + 1, 1), n)
        return float(torch.kthvalue(flat, k).values.item())
    return {
        f'{prefix}_mean_abs': float(flat.mean().item()),
        f'{prefix}_p50_abs': pct(0.50),
        f'{prefix}_p90_abs': pct(0.90),
        f'{prefix}_p99_abs': pct(0.99),
        f'{prefix}_p999_abs': pct(0.999),
        f'{prefix}_max_abs': float(flat.max().item()),
        f'{prefix}_gt_1e-5_ratio': float((flat > 1e-5).float().mean().item()),
        f'{prefix}_gt_1e-4_ratio': float((flat > 1e-4).float().mean().item()),
    }


def compare_face_qef(cpu, other):
    row = {
        'count_cpu': int(cpu['keys'].numel()),
        'count_other': int(other['keys'].numel()),
        'unique_other': other['unique'],
    }
    if not torch.equal(cpu['keys'], other['keys']):
        a = set(cpu['keys'].tolist())
        b = set(other['keys'].tolist())
        row.update({
            'voxel_keys_equal': False,
            'missing': len(a - b),
            'extra': len(b - a),
            'missing_sample': sorted(list(a - b))[:8],
            'extra_sample': sorted(list(b - a))[:8],
        })
        return row
    diff = (other['qefs'] - cpu['qefs']).float()
    ref = cpu['qefs'].float()
    trace_diff = diff[:, 0] + diff[:, 4] + diff[:, 7]
    row.update({
        'voxel_keys_equal': True,
        'missing': 0,
        'extra': 0,
        'qef_rmse': float(torch.sqrt(torch.mean(diff * diff)).item()) if diff.numel() else 0.0,
        'qef_rel_l2': float(torch.linalg.vector_norm(diff).item() / max(torch.linalg.vector_norm(ref).item(), 1e-12)) if diff.numel() else 0.0,
        'qef_cpu_max_abs': float(ref.abs().max().item()) if ref.numel() else 0.0,
        'qef_other_max_abs': float(other['qefs'].abs().max().item()) if other['qefs'].numel() else 0.0,
        'trace_rmse': float(torch.sqrt(torch.mean(trace_diff * trace_diff)).item()) if trace_diff.numel() else 0.0,
        'trace_positive_sum': float(trace_diff[trace_diff > 0].sum().item()) if trace_diff.numel() else 0.0,
        'trace_negative_sum': float(trace_diff[trace_diff < 0].sum().item()) if trace_diff.numel() else 0.0,
    })
    row.update(abs_distribution('qef', diff))
    row.update(abs_distribution('trace', trace_diff))
    return row


def show_rows(rows):
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(row)


def face_old(case, inter):
    return ext.face_qef_old(case['triangles_cuda'], voxel_size_cpu, grid_range_cpu, inter.voxels)


def face_ref(case, inter):
    return ext.face_qef_ref(
        case['triangles_cuda'], voxel_size_cpu, grid_range_cpu,
        inter.voxels, inter.brick_hash_keys, inter.brick_hash_vals, inter.brick_bits, inter.brick_base)


def face_new(case, inter):
    return ext.face_qef(
        case['triangles_cuda'], voxel_size_cpu, grid_range_cpu,
        inter.voxels, inter.brick_hash_keys, inter.brick_hash_vals, inter.brick_bits, inter.brick_base)


def face_cpu(case, inter):
    return ext.face_qef_cpu(case['triangles_cpu'], voxel_size_cpu, grid_range_cpu, inter.voxels.detach().cpu().contiguous())


## Build Active Voxel Inputs

This cell runs only the current `intersect_qef`. Its output supplies the shared `voxels` and brick lookup tensors for all face QEF measurements.

In [5]:
torch.cuda.empty_cache()
torch.cuda.synchronize()

INTERSECT_OUTPUTS = {}
intersect_rows = []
for input_name, case in INPUT_CASES.items():
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    base_alloc = torch.cuda.memory_allocated()
    base_reserved = torch.cuda.memory_reserved()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    out = as_intersect_output(ext.intersect_qef(case['triangles_cuda'], voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES))
    end.record()
    torch.cuda.synchronize()
    INTERSECT_OUTPUTS[input_name] = out
    intersect_rows.append({
        'input': input_name,
        'triangles': case['triangles'],
        'voxels': int(out.voxels.shape[0]),
        'brick_hash_capacity': int(out.brick_hash_keys.shape[0]),
        'brick_slots': int(out.brick_bits.shape[0]),
        'intersect_ms': float(start.elapsed_time(end)),
        'intersect_peak_alloc_mb': float((torch.cuda.max_memory_allocated() - base_alloc) / 1024**2),
        'intersect_peak_reserved_mb': float((torch.cuda.max_memory_reserved() - base_reserved) / 1024**2),
    })

show_rows(intersect_rows)


,input,triangles,voxels,brick_hash_capacity,brick_slots,intersect_ms,intersect_peak_alloc_mb,intersect_peak_reserved_mb
0,original,280333,4676307,524288,262144,26.356159,382.233398,340.0
1,subdivided,10304077,4676306,524288,262144,47.495167,1227.611816,1202.0


## Correctness Against CPU face_qef

All outputs are canonicalized by voxel key before error metrics, so row-order differences do not affect evaluation.

In [6]:
FACE_CANON = {}
correctness_rows = []

for input_name, case in INPUT_CASES.items():
    inter = INTERSECT_OUTPUTS[input_name]
    cpu_allowed = input_name != 'subdivided' or RUN_CPU_SUBDIVIDED
    if not cpu_allowed:
        print(f'skip CPU face_qef for {input_name}; set FDG_RUN_CPU_SUBDIVIDED=1 to enable')
        continue

    print('running CPU face_qef for', input_name, 'triangles:', case['triangles'], 'voxels:', int(inter.voxels.shape[0]))
    cpu_qef = face_cpu(case, inter)
    cpu_can = canonical_face_qef(inter.voxels, cpu_qef)
    FACE_CANON[(input_name, 'cpu')] = cpu_can
    assert cpu_can['unique'], f'{input_name} CPU voxel keys are not unique'

    gpu_outputs = {
        'old': face_old(case, inter),
        'ref': face_ref(case, inter),
        'new': face_new(case, inter),
    }
    torch.cuda.synchronize()

    for method, qef in gpu_outputs.items():
        can = canonical_face_qef(inter.voxels, qef)
        FACE_CANON[(input_name, method)] = can
        row = {'input': input_name, 'method': method, **compare_face_qef(cpu_can, can)}
        correctness_rows.append(row)
        assert row['voxel_keys_equal'], f'{input_name} {method} voxel keys differ from CPU'

show_rows(correctness_rows)


running CPU face_qef for original triangles: 280333 voxels: 4676307
running CPU face_qef for subdivided triangles: 10304077 voxels: 4676306


,input,method,count_cpu,count_other,unique_other,voxel_keys_equal,missing,extra,qef_rmse,qef_rel_l2,...,qef_gt_1e-5_ratio,qef_gt_1e-4_ratio,trace_mean_abs,trace_p50_abs,trace_p90_abs,trace_p99_abs,trace_p999_abs,trace_max_abs,trace_gt_1e-5_ratio,trace_gt_1e-4_ratio
0,original,old,4676307,4676307,True,True,0,0,0.001880,0.002370,...,0.000012,0.000012,0.000017,0.000000e+00,5.960464e-08,4.768372e-07,9.536743e-07,5.000000,0.000015,0.000015
1,original,ref,4676307,4676307,True,True,0,0,0.001508,0.001901,...,0.000012,0.000012,0.000015,0.000000e+00,5.960464e-08,4.768372e-07,9.536743e-07,1.000001,0.000016,0.000015
2,original,new,4676307,4676307,True,True,0,0,0.001533,0.001933,...,0.000012,0.000012,0.000016,0.000000e+00,5.587935e-09,3.725290e-07,9.536743e-07,1.000001,0.000016,0.000016
3,subdivided,old,4676306,4676306,True,True,0,0,0.007169,0.002442,...,0.000093,0.000088,0.000179,3.725290e-09,5.960464e-07,1.907116e-06,3.814697e-06,7.000000,0.000130,0.000124
4,subdivided,ref,4676306,4676306,True,True,0,0,0.003140,0.001069,...,0.000053,0.000050,0.000068,0.000000e+00,2.384186e-07,9.611249e-07,2.145767e-06,2.000000,0.000068,0.000068
5,subdivided,new,4676306,4676306,True,True,0,0,0.003070,0.001046,...,0.000052,0.000049,0.000066,0.000000e+00,1.192093e-07,9.536743e-07,1.907349e-06,2.000000,0.000066,0.000065


## QEF Distribution And Sampled Hit-Set Eval

Full QEF and trace distributions are computed for all voxels. Hit-set TP/FP/FN/F1 is sampled from top trace-error voxels plus random voxels; `old` is omitted because its face candidates come from `voxelize_mesh_octree`, not the aligned face predicate used by `ref/new`.


In [7]:
HIT_EVAL_TOPK = int(os.environ.get('FDG_HIT_EVAL_TOPK', '8'))
HIT_EVAL_RANDOM = int(os.environ.get('FDG_HIT_EVAL_RANDOM', '8'))
HIT_EVAL_METHODS = [m for m in ['ref', 'new']]

trace_distribution_rows = []
hit_candidate_rows = []

for input_name in INPUT_CASES.keys():
    cpu_key = (input_name, 'cpu')
    if cpu_key not in FACE_CANON:
        continue
    cpu = FACE_CANON[cpu_key]
    cpu_trace = cpu['qefs'][:, 0] + cpu['qefs'][:, 4] + cpu['qefs'][:, 7]
    for method in ['old', 'ref', 'new']:
        key = (input_name, method)
        if key not in FACE_CANON:
            continue
        other = FACE_CANON[key]
        if not torch.equal(cpu['keys'], other['keys']):
            continue
        trace_diff = (other['qefs'][:, 0] + other['qefs'][:, 4] + other['qefs'][:, 7] - cpu_trace).float()
        row = {'input': input_name, 'method': method}
        row.update(abs_distribution('trace', trace_diff))
        row['trace_rmse'] = float(torch.sqrt(torch.mean(trace_diff * trace_diff)).item()) if trace_diff.numel() else 0.0
        row['trace_positive_sum'] = float(trace_diff[trace_diff > 0].sum().item()) if trace_diff.numel() else 0.0
        row['trace_negative_sum'] = float(trace_diff[trace_diff < 0].sum().item()) if trace_diff.numel() else 0.0
        trace_distribution_rows.append(row)

        if method not in HIT_EVAL_METHODS:
            continue
        abs_trace = trace_diff.abs()
        if abs_trace.numel() == 0:
            continue
        topk = min(HIT_EVAL_TOPK, int(abs_trace.numel()))
        if topk:
            _, idx = torch.topk(abs_trace, topk)
            for rank, row_id in enumerate(idx.tolist()):
                hit_candidate_rows.append({
                    'input': input_name,
                    'method': method,
                    'sample': 'top_trace_abs',
                    'rank': rank,
                    'row_id': int(row_id),
                    'voxel_key': int(cpu['keys'][row_id].item()),
                    'voxel': tuple(int(v) for v in cpu['voxels'][row_id].tolist()),
                    'trace_diff': float(trace_diff[row_id].item()),
                    'trace_abs': float(abs_trace[row_id].item()),
                    'qef_max_abs_voxel': float((other['qefs'][row_id] - cpu['qefs'][row_id]).abs().max().item()),
                })
        randk = min(HIT_EVAL_RANDOM, int(abs_trace.numel()))
        if randk:
            gen = torch.Generator().manual_seed(20260627 + len(hit_candidate_rows))
            idx = torch.randperm(abs_trace.numel(), generator=gen)[:randk]
            for rank, row_id in enumerate(idx.tolist()):
                hit_candidate_rows.append({
                    'input': input_name,
                    'method': method,
                    'sample': 'random',
                    'rank': rank,
                    'row_id': int(row_id),
                    'voxel_key': int(cpu['keys'][row_id].item()),
                    'voxel': tuple(int(v) for v in cpu['voxels'][row_id].tolist()),
                    'trace_diff': float(trace_diff[row_id].item()),
                    'trace_abs': float(abs_trace[row_id].item()),
                    'qef_max_abs_voxel': float((other['qefs'][row_id] - cpu['qefs'][row_id]).abs().max().item()),
                })

show_rows(trace_distribution_rows)
show_rows(hit_candidate_rows[:32])

hit_cache = {}

def probe_hit_sets(input_name, voxel_tuple):
    key = (input_name, voxel_tuple)
    if key in hit_cache:
        return hit_cache[key]
    case = INPUT_CASES[input_name]
    voxel = torch.tensor(voxel_tuple, dtype=torch.int32)
    cpu_hit = ext.face_qef_cpu_hit_probe(case['triangles_cpu'], voxel_size_cpu, grid_range_cpu, voxel).bool()
    gpu_hit = ext.face_qef_gpu_hit_probe(case['triangles_cuda'], voxel_size_cpu, grid_range_cpu, voxel).detach().cpu().bool()
    torch.cuda.synchronize()
    hit_cache[key] = (cpu_hit, gpu_hit)
    return hit_cache[key]

hit_rows = []
for cand in hit_candidate_rows:
    cpu_hit, gpu_hit = probe_hit_sets(cand['input'], cand['voxel'])
    tp = int((cpu_hit & gpu_hit).sum().item())
    fp = int((~cpu_hit & gpu_hit).sum().item())
    fn = int((cpu_hit & ~gpu_hit).sum().item())
    cpu_count = int(cpu_hit.sum().item())
    gpu_count = int(gpu_hit.sum().item())
    precision = tp / (tp + fp) if (tp + fp) else 1.0
    recall = tp / (tp + fn) if (tp + fn) else 1.0
    f1 = 2.0 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    hit_rows.append({
        **cand,
        'cpu_hits': cpu_count,
        'gpu_hits': gpu_count,
        'true_pos': tp,
        'false_pos': fp,
        'false_neg': fn,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    })

summary = {}
for row in hit_rows:
    key = (row['input'], row['method'], row['sample'])
    summary.setdefault(key, []).append(row)

hit_summary_rows = []
for (input_name, method, sample), rows in summary.items():
    n = len(rows)
    hit_summary_rows.append({
        'input': input_name,
        'method': method,
        'sample': sample,
        'voxels': n,
        'avg_cpu_hits': sum(r['cpu_hits'] for r in rows) / n,
        'avg_gpu_hits': sum(r['gpu_hits'] for r in rows) / n,
        'avg_true_pos': sum(r['true_pos'] for r in rows) / n,
        'avg_false_pos': sum(r['false_pos'] for r in rows) / n,
        'avg_false_neg': sum(r['false_neg'] for r in rows) / n,
        'max_false_pos': max(r['false_pos'] for r in rows),
        'max_false_neg': max(r['false_neg'] for r in rows),
        'avg_precision': sum(r['precision'] for r in rows) / n,
        'avg_recall': sum(r['recall'] for r in rows) / n,
        'avg_f1': sum(r['f1'] for r in rows) / n,
    })

show_rows(hit_summary_rows)
show_rows(hit_rows[:64])


,input,method,trace_mean_abs,trace_p50_abs,trace_p90_abs,trace_p99_abs,trace_p999_abs,trace_max_abs,trace_gt_1e-5_ratio,trace_gt_1e-4_ratio,trace_rmse,trace_positive_sum,trace_negative_sum
0,original,old,0.000017,0.0,0.000000e+00,4.768372e-07,9.536743e-07,5.000000,0.000015,0.000015,0.004894,42.046165,-36.045975
1,original,ref,0.000015,0.0,0.000000e+00,4.768372e-07,9.536743e-07,1.000001,0.000016,0.000015,0.003924,33.047256,-39.046719
2,original,new,0.000016,0.0,0.000000e+00,4.768372e-07,9.536743e-07,1.000001,0.000016,0.000016,0.003951,41.040703,-32.040527
3,subdivided,old,0.000179,0.0,9.536743e-07,1.907349e-06,3.814697e-06,7.000000,0.000133,0.000124,0.018716,644.412415,-194.414032
4,subdivided,ref,0.000068,0.0,0.000000e+00,9.536743e-07,3.814697e-06,2.000001,0.000069,0.000068,0.008298,124.141541,-194.140121
5,subdivided,new,0.000066,0.0,0.000000e+00,9.536743e-07,1.907349e-06,2.000001,0.000066,0.000065,0.008168,120.106041,-188.104263


,input,method,sample,rank,row_id,voxel_key,voxel,trace_diff,trace_abs,qef_max_abs_voxel
0,original,ref,top_trace_abs,0,4384949,82226434,"(258, 342, 313)",1.000001e+00,1.000001e+00,9.540944e-01
1,original,ref,top_trace_abs,1,2439961,69534791,"(71, 130, 265)",-1.000000e+00,1.000000e+00,6.259048e-01
2,original,ref,top_trace_abs,2,4285303,78727610,"(442, 164, 300)",-1.000000e+00,1.000000e+00,4.039147e-01
3,original,ref,top_trace_abs,3,1095600,63508884,"(404, 136, 242)",1.000000e+00,1.000000e+00,6.156588e-01
4,original,ref,top_trace_abs,4,1965491,67579681,"(289, 407, 257)",-1.000000e+00,1.000000e+00,6.083976e-01
5,original,ref,top_trace_abs,5,2763770,70844309,"(405, 127, 270)",1.000000e+00,1.000000e+00,7.500038e-01
6,original,ref,top_trace_abs,6,3813700,75590731,"(75, 182, 288)",-1.000000e+00,1.000000e+00,4.656566e-01
7,original,ref,top_trace_abs,7,941124,62600426,"(234, 410, 238)",1.000000e+00,1.000000e+00,1.071303e+00
8,original,ref,random,0,351588,42930006,"(342, 391, 163)",0.000000e+00,0.000000e+00,1.490116e-08
9,original,ref,random,1,2464698,69615543,"(439, 287, 265)",0.000000e+00,0.000000e+00,1.136868e-13


,input,method,sample,voxels,avg_cpu_hits,avg_gpu_hits,avg_true_pos,avg_false_pos,avg_false_neg,max_false_pos,max_false_neg,avg_precision,avg_recall,avg_f1
0,original,ref,top_trace_abs,8,5.375,5.375,4.875,0.500,0.500,1,1,0.887500,0.891667,0.872710
1,original,ref,random,8,2.750,2.750,2.750,0.000,0.000,0,0,1.000000,1.000000,1.000000
2,original,new,top_trace_abs,8,4.125,4.375,4.000,0.375,0.125,1,1,0.875000,0.979167,0.910606
3,original,new,random,8,1.625,1.625,1.625,0.000,0.000,0,0,1.000000,1.000000,1.000000
4,subdivided,ref,top_trace_abs,8,26.625,26.875,25.625,1.250,1.000,8,3,0.967496,0.927815,0.942003
5,subdivided,ref,random,8,8.750,8.750,8.750,0.000,0.000,0,0,1.000000,1.000000,1.000000
6,subdivided,new,top_trace_abs,8,30.125,29.750,29.125,0.625,1.000,3,3,0.986246,0.928350,0.952965
7,subdivided,new,random,8,13.375,13.375,13.375,0.000,0.000,0,0,1.000000,1.000000,1.000000


,input,method,sample,rank,row_id,voxel_key,voxel,trace_diff,trace_abs,qef_max_abs_voxel,cpu_hits,gpu_hits,true_pos,false_pos,false_neg,precision,recall,f1
0,original,ref,top_trace_abs,0,4384949,82226434,"(258, 342, 313)",1.000001,1.000001,9.540944e-01,14,15,14,1,0,0.933333,1.000000,0.965517
1,original,ref,top_trace_abs,1,2439961,69534791,"(71, 130, 265)",-1.000000,1.000000,6.259048e-01,6,5,5,0,1,1.000000,0.833333,0.909091
2,original,ref,top_trace_abs,2,4285303,78727610,"(442, 164, 300)",-1.000000,1.000000,4.039147e-01,6,5,5,0,1,1.000000,0.833333,0.909091
3,original,ref,top_trace_abs,3,1095600,63508884,"(404, 136, 242)",1.000000,1.000000,6.156588e-01,5,6,5,1,0,0.833333,1.000000,0.909091
4,original,ref,top_trace_abs,4,1965491,67579681,"(289, 407, 257)",-1.000000,1.000000,6.083976e-01,3,2,2,0,1,1.000000,0.666667,0.800000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,subdivided,new,random,3,32636,30451385,"(185, 83, 116)",0.000000,0.000000,5.960464e-08,8,8,8,0,0,1.000000,1.000000,1.000000
60,subdivided,new,random,4,472053,46815434,"(202, 300, 178)",0.000000,0.000000,2.384186e-07,12,12,12,0,0,1.000000,1.000000,1.000000
61,subdivided,new,random,5,2948822,71625453,"(237, 117, 273)",0.000000,0.000000,4.768372e-07,28,28,28,0,0,1.000000,1.000000,1.000000
62,subdivided,new,random,6,3183881,72652999,"(199, 76, 277)",0.000000,0.000000,5.960464e-08,9,9,9,0,0,1.000000,1.000000,1.000000


## GPU Timing And Memory

Cold runs clear the caching allocator before the call. Warm results report the median across repeats. Memory numbers are peak deltas relative to the live inputs from `intersect_qef`.

In [8]:
def timed_cuda_call(fn, cold=False):
    if cold:
        torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    base_alloc = torch.cuda.memory_allocated()
    base_reserved = torch.cuda.memory_reserved()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    out = fn()
    end.record()
    torch.cuda.synchronize()
    ms = start.elapsed_time(end)
    peak_alloc = torch.cuda.max_memory_allocated() - base_alloc
    peak_reserved = torch.cuda.max_memory_reserved() - base_reserved
    del out
    torch.cuda.synchronize()
    return {
        'ms': float(ms),
        'peak_alloc_mb': float(peak_alloc / 1024**2),
        'peak_reserved_mb': float(peak_reserved / 1024**2),
    }


def summarize_records(records):
    return {
        'ms_median': statistics.median(r['ms'] for r in records),
        'ms_min': min(r['ms'] for r in records),
        'peak_alloc_mb_median': statistics.median(r['peak_alloc_mb'] for r in records),
        'peak_reserved_mb_median': statistics.median(r['peak_reserved_mb'] for r in records),
    }


bench_rows = []
for input_name, case in INPUT_CASES.items():
    inter = INTERSECT_OUTPUTS[input_name]
    methods = {
        'old': lambda case=case, inter=inter: face_old(case, inter),
        'ref': lambda case=case, inter=inter: face_ref(case, inter),
        'new': lambda case=case, inter=inter: face_new(case, inter),
    }
    for method, fn in methods.items():
        cold = timed_cuda_call(fn, cold=True)
        _ = timed_cuda_call(fn, cold=False)
        warm_records = [timed_cuda_call(fn, cold=False) for _ in range(WARM_REPEATS)]
        warm = summarize_records(warm_records)
        bench_rows.append({
            'input': input_name,
            'triangles': case['triangles'],
            'voxels': int(inter.voxels.shape[0]),
            'input_triangle_mb': case['triangle_mb'],
            'method': method,
            'cold_ms': cold['ms'],
            'cold_peak_alloc_mb': cold['peak_alloc_mb'],
            'cold_peak_reserved_mb': cold['peak_reserved_mb'],
            **warm,
        })

show_rows(bench_rows)


,input,triangles,voxels,input_triangle_mb,method,cold_ms,cold_peak_alloc_mb,cold_peak_reserved_mb,ms_median,ms_min,peak_alloc_mb_median,peak_reserved_mb_median
0,original,280333,4676307,9.624470,old,30.025984,1552.552246,1108.0,24.597392,24.304352,1553.522461,0.0
1,original,280333,4676307,9.624470,ref,28.329985,178.387207,180.0,27.667104,26.948481,178.387207,0.0
2,original,280333,4676307,9.624470,new,4.533408,204.424805,182.0,2.737664,2.695872,204.424805,0.0
3,subdivided,10304077,4676306,353.762409,old,321.077240,17077.544922,19270.0,313.588058,310.308990,17077.544922,0.0
4,subdivided,10304077,4676306,353.762409,ref,16.639999,178.387207,180.0,15.891296,15.590400,178.387207,0.0
5,subdivided,10304077,4676306,353.762409,new,21.089376,561.791992,408.0,17.590976,17.065792,561.791992,0.0
